In [1]:
import sys
import os
import torch

In [2]:
module_path = "/home/ubuntu/Shree_FYP/train/stage2/models"

In [3]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
import final_latent_student

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
from final_verbalizer import Verbalizer

In [6]:
v = Verbalizer(model_name="unsloth/Qwen3.5-0.8B", student_hidden=2560)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [7]:
v.lm.model.model.config.text_config.hidden_size

1024

In [8]:
layer = v.lm.model.model.language_model.layers[0]

In [9]:
def inspect_hook(module, args, output):
    print(f"🔍 Layer 0 return type: {type(output).__name__}")
    
    if isinstance(output, tuple):
        print(f"   ✅ It's a TUPLE (length={len(output)})")
        print(f"   → hidden_states shape: {output[0].shape}")
        if len(output) > 1:
            print(f"   → Extra elements: {[type(x).__name__ for x in output[1:]]}")
    elif isinstance(output, torch.Tensor):
        print(f"   ⚠️  It's a DIRECT TENSOR")
        print(f"   → Shape: {output.shape}")
    else:
        # Rare: HuggingFace output object (e.g., BaseModelOutputWithPast)
        print(f"   ⚠️  It's an HF OUTPUT OBJECT")
        if hasattr(output, "last_hidden_state"):
            print(f"   → .last_hidden_state shape: {output.last_hidden_state.shape}")
        elif hasattr(output, "hidden_states"):
            print(f"   → .hidden_states shape: {output.hidden_states[0].shape}")
            
    return output  # ⚠️ Must return output unchanged!

In [10]:
handle = layer.register_forward_hook(inspect_hook)

In [11]:
dummy_ids = torch.randint(0, 200, (1, 4), device="cuda")
mask = torch.ones_like(dummy_ids)
embeds = v._embed_tokens(dummy_ids)

In [12]:
with torch.no_grad():
    # Return_dict doesn't affect raw layer hooks, but we keep it explicit
    v._language_model(inputs_embeds=embeds, attention_mask=mask, return_dict=False)

🔍 Layer 0 return type: Tensor
   ⚠️  It's a DIRECT TENSOR
   → Shape: torch.Size([1, 4, 1024])


In [13]:
handle.remove()
print("✅ Hook removed. Test complete.")

✅ Hook removed. Test complete.


In [14]:
# 2. Dummy inputs
dummy_ids = torch.randint(0, 200, (1, 4), device="cuda")
mask = torch.ones_like(dummy_ids)
dummy_latents = torch.randn(1, 6, 2560, device="cuda", dtype=torch.bfloat16)

In [15]:
v.to("cuda")

Verbalizer(
  (lm): PeftModelForCausalLM(
    (base_model): LoraModel(
      (model): Qwen3_5ForConditionalGeneration(
        (model): Qwen3_5Model(
          (visual): Qwen3_5VisionModel(
            (patch_embed): Qwen3_5VisionPatchEmbed(
              (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
            )
            (pos_embed): Embedding(2304, 768)
            (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
            (blocks): ModuleList(
              (0-11): 12 x Qwen3_5VisionBlock(
                (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
                (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
                (attn): Qwen3_5VisionAttention(
                  (qkv): Linear(in_features=768, out_features=2304, bias=True)
                  (proj): Linear(in_features=768, out_features=768, bias=True)
                )
                (mlp): Qwen3_5VisionMLP(
                  (linear_fc1): Linear(in_features=

In [16]:
print("🚀 Running Verbalizer forward pass with latent injection...")
with torch.no_grad():
    logits, loss = v._lm_forward(
        input_ids=dummy_ids,
        attention_mask=mask,
        latents=dummy_latents,
        labels=None
    )
print(f"✅ Success! Logits shape: {logits.shape}")  # Expected: [1, 4, 248320]
print("🎉 Safe-unpack is working. CA injection completed without shape corruption.")

🚀 Running Verbalizer forward pass with latent injection...
✅ Success! Logits shape: torch.Size([1, 4, 248320])
🎉 Safe-unpack is working. CA injection completed without shape corruption.


In [17]:
print(f"✅ Success! Logits shape: {logits.shape}")  # Expected: [1, 4, 248320]
print("🎉 Safe-unpack is working. CA injection completed without shape corruption.")

✅ Success! Logits shape: torch.Size([1, 4, 248320])
🎉 Safe-unpack is working. CA injection completed without shape corruption.
